In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
categories = ["rec.autos", "sci.space", "sci.med"]

train_data = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")  # removes obvious metadata
)

test_data = fetch_20newsgroups(
    subset="test",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

X_train_raw, y_train = train_data.data, train_data.target
X_test_raw, y_test = test_data.data, test_data.target
target_names = train_data.target_names

print("Train size:", len(X_train_raw))
print("Test size :", len(X_test_raw))
print("Classes   :", target_names)

Train size: 1781
Test size : 1186
Classes   : ['rec.autos', 'sci.med', 'sci.space']


In [3]:
def clean_text(text: str) -> str:
    # 1) lowercase
    text = text.lower()

    # 2) remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # 3) keep only letters (replace everything else with space)
    text = re.sub(r"[^a-z\s]", " ", text)

    # 4) tokenize
    tokens = text.split()

    # 5) remove stopwords + short tokens
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 2]

    # return cleaned text (space-separated tokens)
    return " ".join(tokens)

# quick demo: see before/after on one sample
sample = X_train_raw[0]
print("BEFORE:\n", sample[:400], "\n")
print("AFTER:\n", clean_text(sample)[:400])

BEFORE:
 
A(>  Can anyone tell me if a bloodcount of 40 when diagnosed as hypoglycemic is
A(>  dangerous, i.e. indicates a possible pancreatic problem?  One Dr. says no, the
A(>  other (not his specialty) says the first is negligent and that another blood
A(>  test should be done.  Also, what is a good diet (what has worked) for a hypo-
A(>  glycemic?  TIA.
A(>  
A(>  
A(>  Anthony Anello
A(>  Fermilab
A(> 

AFTER:
 tell bloodcount diagnosed hypoglycemic dangerous indicates possible pancreatic problem says specialty says negligent blood test good diet worked hypo glycemic tia anthony anello fermilab batavia illinois hypoglycemia confirmed proper channels consider ther following chelated manganese day chelated chromium mcg day increase protein foods supplements avoid supplements foods high potassium calcium zi


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Pipeline = run multiple steps in order (text -> numbers -> classifier)
model = Pipeline([
    # Step 1: TF-IDF Vectorizer (converts text into numeric feature vectors)
    #
    # What TF-IDF does:
    # - It builds a "vocabulary" of words from the training text.
    # - For each document, it creates a long vector where each position corresponds to a word.
    # - The value at each position is the TF-IDF score of that word in that document.
    #
    # TF (Term Frequency): how often a word appears in a document.
    # IDF (Inverse Document Frequency): reduces the weight of common words
    #   (words that appear in many documents get lower weight).
    #
    # Result: each document becomes a numeric vector that represents "important words".
    #
    # preprocessor=clean_text means: before vectorizing, apply our cleaning function
    # (lowercase, remove punctuation, remove stopwords, etc.)
    #
    # min_df=2 means: ignore words that appear in fewer than 2 documents
    # (removes very rare words/noise).
    ("tfidf", TfidfVectorizer(preprocessor=clean_text, min_df=2)),

    # Step 2: Multinomial Naive Bayes classifier
    #
    # This model learns:
    # - which words are common in each class (cars vs space vs medical)
    # - then predicts the class of a new document using those learned word probabilities.
    ("nb", MultinomialNB())
])

# Train the pipeline:
# - tfidf step learns vocabulary + transforms training text into TF-IDF vectors
# - nb step learns class probabilities from those vectors
model.fit(X_train_raw, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(min_df=2,
                                 preprocessor=<function clean_text at 0x7fb0636f5580>)),
                ('nb', MultinomialNB())])

In [5]:
y_pred = model.predict(X_test_raw)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8988195615514334
